In [3]:
import pandas as pd

loans = pd.read_csv(
    "data/library_loans_clean.csv",
    parse_dates=["loan_date", "return_date"]
)

print("Shape of the dataset:", loans.shape)
print(loans.dtypes)

missing = pd.DataFrame({
    "missing_count": loans.isna().sum(),
    "missing_percent": (loans.isna().mean() * 100).round(2),
})

print(missing)

Shape of the dataset: (600, 8)
loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object
             missing_count  missing_percent
loan_id                  0              0.0
member_id                0              0.0
title                    0              0.0
genre                    0              0.0
loan_date                3              0.5
return_date             36              6.0
late_fee                 0              0.0
returned                 0              0.0


In [5]:
# Show loans with a missing loan_date
loans.loc[loans["loan_date"].isna()]

# Count missing return_date by whether the book was returned
missing_return = loans["return_date"].isna()

print("Missing return_date, returned=True:",
      (missing_return & loans["returned"].eq(True)).sum())
print("Missing return_date, returned=False:",
      (missing_return & loans["returned"].eq(False)).sum())

# Missing because the book is still out: returned=False
# Missing by accident: returned=True

loans["has_return_date"] = loans["return_date"].notna()
print(loans["has_return_date"].value_counts())

loans.loc[loans["loan_date"].isna()]

# Count missing return_date by whether the book was returned
missing_return = loans["return_date"].isna()

print("Missing return_date, returned=True:",
      (missing_return & loans["returned"].eq(True)).sum())
print("Missing return_date, returned=False:",
      (missing_return & loans["returned"].eq(False)).sum())

# Missing because the book is still out: returned=False
# Missing by accident: returned=True

loans["has_return_date"] = loans["return_date"].notna()
print(loans["has_return_date"].value_counts())

Missing return_date, returned=True: 6
Missing return_date, returned=False: 30
has_return_date
True     564
False     36
Name: count, dtype: int64
Missing return_date, returned=True: 6
Missing return_date, returned=False: 30
has_return_date
True     564
False     36
Name: count, dtype: int64


In [6]:
members = pd.read_csv("data/library_members.csv").copy()

print("Shape:", members.shape)
members.info()
print(members["birth_year"].describe())

# Count the values at the minimum and maximum
min_year = members["birth_year"].min()
max_year = members["birth_year"].max()
print("Minimum birth_year:", min_year,
      "count:", members["birth_year"].eq(min_year).sum())
print("Maximum birth_year:", max_year,
      "count:", members["birth_year"].eq(max_year).sum())

# Check duplicate rows and duplicate member IDs
print("Duplicate rows:", members.duplicated().sum())
print("Duplicate member_id values:", members["member_id"].duplicated().sum())

Shape: (148, 3)
<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.6 KB
count     130.000000
mean     1983.384615
std        25.158223
min      1900.000000
25%      1978.250000
50%      1988.000000
75%      1996.750000
max      2030.000000
Name: birth_year, dtype: float64
Minimum birth_year: 1900.0 count: 8
Maximum birth_year: 2030.0 count: 3
Duplicate rows: 4
Duplicate member_id values: 4


In [7]:
members["birth_year"].value_counts().sort_index().head(10), \
members["birth_year"].value_counts().sort_index().tail(10)

(birth_year
 1900.0    8
 1957.0    1
 1960.0    1
 1961.0    1
 1962.0    1
 1967.0    2
 1968.0    3
 1970.0    2
 1971.0    2
 1974.0    2
 Name: count, dtype: int64,
 birth_year
 2001.0    3
 2002.0    1
 2003.0    3
 2004.0    3
 2005.0    3
 2007.0    1
 2008.0    1
 2009.0    1
 2010.0    4
 2030.0    3
 Name: count, dtype: int64)

In [8]:
members["birth_year"] = members["birth_year"].where(
    members["birth_year"].between(1920, 2015)
)

print("Missing birth years:", members["birth_year"].isna().sum())
print(members["birth_year"].describe())

Missing birth years: 29
count     119.000000
mean     1987.815126
std        12.062666
min      1957.000000
25%      1979.000000
50%      1988.000000
75%      1996.500000
max      2010.000000
Name: birth_year, dtype: float64


In [9]:
print("After dropna():", len(members.dropna()))
print("After dropna(subset=['branch']):", len(members.dropna(subset=["branch"])))
print("After dropna(subset=['birth_year']):", len(members.dropna(subset=["birth_year"])))

After dropna(): 107
After dropna(subset=['branch']): 136
After dropna(subset=['birth_year']): 119


In [10]:
members["branch"] = members["branch"].fillna("Unknown")
print(members["branch"].value_counts())

branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64


In [11]:
birth_year_filled = members["birth_year"].fillna(members["birth_year"].median())

print("Missing values after filling:", birth_year_filled.isna().sum())
print(birth_year_filled.describe())

Missing values after filling: 0
count     148.000000
mean     1987.851351
std        10.807760
min      1957.000000
25%      1982.000000
50%      1988.000000
75%      1994.000000
max      2010.000000
Name: birth_year, dtype: float64


In [12]:
print("Loans shape:", loans.shape)
print("Members shape:", members.shape)

loans_with_members = loans.merge(members, on="member_id", how="left")
print("Merged shape:", loans_with_members.shape)

Loans shape: (600, 9)
Members shape: (148, 3)
Merged shape: (616, 11)


In [13]:
members = members.drop_duplicates(subset="member_id")

loans_with_members = loans.merge(
    members,
    on="member_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Merged shape:", loans_with_members.shape)
print(loans_with_members["_merge"].value_counts())

Merged shape: (600, 12)
_merge
both          569
left_only      31
right_only      0
Name: count, dtype: int64
